# Step 07: Chat with Multiple Plugins (Alternative)

This is similar to Step 06, demonstrating interactive chat with location and time context.

In [1]:
import asyncio
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv, find_dotenv
dotenv_path = find_dotenv()
load_dotenv(dotenv_path, override=True)

True

In [3]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import (
    AzureChatCompletion,
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.contents.chat_history import ChatHistory
from plugins.datetime import DateTimePlugin
from plugins.location import LocationPlugin

In [4]:
# Initialize kernel and plugins
kernel = Kernel()
azure_chat_completion = AzureChatCompletion()
kernel.add_service(azure_chat_completion)

kernel.add_plugin(LocationPlugin(), plugin_name="location_plugin")
kernel.add_plugin(DateTimePlugin(), plugin_name="time_plugin")

execution_settings = AzureChatPromptExecutionSettings()

In [5]:
# Get location and time context
location_function = kernel.get_function(
    plugin_name="location_plugin", function_name="GetLocationInfo"
)
location_info = await kernel.invoke(location_function)

time = kernel.get_function(plugin_name="time_plugin", function_name="Time")
current_time = await kernel.invoke(time)

print(f"Location: {location_info}")
print(f"Time: {current_time}")

Location: Athlone, Leinster, IE
Coordinates: 53.4228,-7.9372
Timezone: Europe/Dublin
Time: 03:31:31 PM


In [6]:
# Initialize chat history with context
history = ChatHistory()
history.add_system_message("User location: " + str(location_info))
history.add_system_message("Time that User started this chat: " + str(current_time))

In [7]:
# Interactive chat function
async def chat(user_input: str):
    history.add_user_message(user_input)
    
    result = await azure_chat_completion.get_chat_message_content(
        chat_history=history,
        settings=execution_settings,
        kernel=kernel,
    )
    
    print("Assistant > " + str(result))
    history.add_message(result)
    return result

In [8]:
# Example chat interaction
await chat("Hello! Can you help me?")

Assistant > Of course! How can I assist you today?


ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-Ds84MWgRlUaiF9gd1N1sX1TqO74Hk', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Of course! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1781793102, model='gpt-4o-2024-11-20', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_49e2bef596', usage=CompletionUsage(completion_tokens=11, prompt_tokens=66, total_tokens=77, completion_tokens_details=CompletionTokensDetails(accepted_prediction_